In [1]:
import logging
import tempfile
from pathlib import Path

from dotenv import load_dotenv

import databao.agent as bao
from databao.agent.configs.agent import AgentConfig
from databao.agent.databases import DuckDBConnectionConfig
from databao.agent.executors import ClaudeAgentExecutor, ClaudeCodeExecutor
from databao.agent.executors.query_expansion import QueryExpansionConfig

load_dotenv()

logging.basicConfig(level=logging.INFO)

EXAMPLES_DIR = Path.cwd()
# NOTE: (@gas) in order to build the context with DCE,
# dbt project should be "initialized", e.g. with `dbt run`;
# the demo project is taken from the Spider-2-dbt dataset
DBT_PROJ_PATH = EXAMPLES_DIR / "shopify002"
DB_PATH = DBT_PROJ_PATH / "shopify.duckdb"
DOMAIN_PATH = "/Users/andrei.gasparian/Documents/databao-project/databao/domains/root"

In [2]:
llm_config = bao.LLMConfig(name="claude-opus-4-6", temperature=0)
agent_config = AgentConfig(recursion_limit=100, parallel_tool_calls=True)

In [3]:
domain_ctx = bao.domain(project_dir=DOMAIN_PATH)

In [4]:
agent = bao.agent(
    domain=domain_ctx,
    name="demo-dbt-executor",
    llm_config=llm_config,
    agent_config=agent_config,
    data_executor=ClaudeAgentExecutor(),
    # data_executor=ClaudeCodeExecutor(),
)

In [5]:
thread = agent.thread(stream_ask=True)

In [6]:
thread.ask(
    "What is our refund rate by month?",
    metadata={"source": "shopify_dbt"},
)

======== <THINKING> ========



INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Users/andrei.gasparian/Documents/databao-agent/.venv/lib/python3.12/site-packages/claude_agent_sdk/_bundled/claude




[tool_call: 'Agent']
```
{'description': 'Retrieve refund rate schema context', 'subagent_type': 'schema-and-context-retriever', 'prompt': 'I need to calculate the refund rate by month. Please find relevant schema, tables, columns, and metric definitions related to:\n- Refunds\n- Orders or transactions\n- Refund rate metric definitions\nReturn all relevant table names, column names, and any SQL hints or metric formulas.'}
```



[tool_call: 'Glob']
```
{'pattern': '/Users/andrei.gasparian/Documents/databao-agent/examples/shopify002/models/**/*.sql'}
```

[tool_call_output: 'Glob']
```
/Users/andrei.gasparian/Documents/databao-agent/examples/shopify002/models/standardized_models/shopify__line_item_enhanced.sql
/Users/andrei.gasparian/Documents/databao-agent/examples/shopify002/models/shopify__inventory_levels.sql
/Users/andrei.gasparian/Documents/databao-agent/examples/shopify002/models/utils/shopify__calendar.sql
/Users/andrei.gasparian/Documents/databao-agent/examples/shopify002/mod

,month,total_orders,refunded_orders,refund_rate_order_pct,refunded_amount,gross_sales,refund_rate_amount_pct
0,2020-09-01,193,2.0,1.04,68.0,29433.19,0.23
1,2020-10-01,310,2.0,0.65,126.4,49248.23,0.26
2,2020-11-01,300,2.0,0.67,94.0,43816.56,0.21
3,2020-12-01,310,2.0,0.65,94.0,50155.70,0.19
4,2021-01-01,310,2.0,0.65,108.0,45482.19,0.24
5,2021-02-01,280,2.0,0.71,84.2,46161.89,0.18
6,2021-03-01,110,2.0,1.82,54.0,15161.07,0.36


In [8]:
print("\n=== TEXT ===\n")
print(thread.text())


=== TEXT ===

**Refund Rate by Month** (Sep 2020 – Mar 2021)

Two refund rate metrics are shown:
- **Order refund rate** — share of orders that had any refund (by count)
- **Amount refund rate** — refunded $ as a share of gross sales (by revenue)

Overall, both rates are very low. The order refund rate ranges from **0.65% to 1.82%**, and the amount refund rate stays between **0.18% and 0.36%**. March 2021 is an outlier with a notably higher order refund rate (1.82%) and amount refund rate (0.36%), likely driven by the lower order volume that month (110 orders vs. ~300 in prior months).


In [9]:
print("\n=== CODE ===\n")
print(thread.code())


=== CODE ===


SELECT
    month,
    total_orders,
    refunded_orders,
    ROUND(refund_rate_order_share * 100, 2)  AS refund_rate_order_pct,
    ROUND(refunded_amount, 2)                AS refunded_amount,
    ROUND(gross_sales, 2)                    AS gross_sales,
    ROUND(refund_rate_amount_share * 100, 2) AS refund_rate_amount_pct
FROM shopify.fct_monthly_refund_rate
ORDER BY month



In [10]:
print("\n=== Dataframe ===\n")
print(thread.df())


=== Dataframe ===

       month  total_orders  refunded_orders  refund_rate_order_pct  \
0 2020-09-01           193              2.0                   1.04   
1 2020-10-01           310              2.0                   0.65   
2 2020-11-01           300              2.0                   0.67   
3 2020-12-01           310              2.0                   0.65   
4 2021-01-01           310              2.0                   0.65   
5 2021-02-01           280              2.0                   0.71   
6 2021-03-01           110              2.0                   1.82   

   refunded_amount  gross_sales  refund_rate_amount_pct  
0             68.0     29433.19                    0.23  
1            126.4     49248.23                    0.26  
2             94.0     43816.56                    0.21  
3             94.0     50155.70                    0.19  
4            108.0     45482.19                    0.24  
5             84.2     46161.89                    0.18  
6            